# TBP-02 Resource Addendum: REST API, WebSockets, and Corrected Offline RSI

This additive notebook keeps all live broker scripts as source references and runs only offline examples. It covers the in-class helper library, endpoint mapping, WebSocket message shape, heartbeat discipline, and a corrected RSI decision table using the local `historical_sample.csv` file.

## 1. Load the additive reference files

The source scripts were copied under new `*_source` filenames. The tables below map those live examples into offline-safe practice objects.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

code_dir = Path.cwd()
inventory = pd.read_csv(code_dir / "tbp02_source_file_inventory.csv")
endpoints = pd.read_csv(code_dir / "tbp02_endpoint_reference.csv")
status_ref = pd.read_csv(code_dir / "tbp02_status_code_reference.csv")
channels = pd.read_csv(code_dir / "tbp02_rest_websocket_reference.csv")

print("Source files copied:", len(inventory[inventory["added_copy"] != "not copied"]))
print("Endpoint/channel rows:", len(endpoints))
print(endpoints[["function", "method", "endpoint"]].head(10).to_string(index=False))

Source files copied: 8
Endpoint/channel rows: 19
              function method                        endpoint
          validate_sso    GET                    sso/validate
                logout   POST                          logout
        reauthenticate   POST          iserver/reauthenticate
           auth_status   POST             iserver/auth/status
                tickle   POST                          tickle
fetch_contract_details    GET   iserver/contract/{conid}/info
 fetch_historical_data    GET      iserver/marketdata/history
             fetch_ltp    GET     iserver/marketdata/snapshot
        fetch_accounts    GET                iserver/accounts
       get_account_pnl    GET iserver/account/pnl/partitioned


## 2. Status-first REST request handling

The lecture rule is to read the HTTP status family first, then parse content only when the response is usable. The mock below includes success, client error, and server error cases without making any network request.

In [2]:
class OfflineResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.content = json.dumps(payload).encode("utf-8")
    def json(self):
        return json.loads(self.content)

def interpret_response(operation, response):
    family = f"{str(response.status_code)[0]}xx"
    payload = response.json()
    if 200 <= response.status_code < 300:
        action = "parse content"
    elif 400 <= response.status_code < 500:
        action = "fix client request or authentication"
    elif response.status_code >= 500:
        action = "log, pause, and retry only under policy"
    else:
        action = "check broker documentation"
    return {"operation": operation, "status_code": response.status_code, "family": family, "client_action": action, "content": payload}

responses = [
    ("auth_status", OfflineResponse(200, {"authenticated": True, "connected": True})),
    ("bad_endpoint", OfflineResponse(404, {"error": "page not found"})),
    ("gateway_down", OfflineResponse(503, {"error": "gateway unavailable"})),
]
status_demo = pd.DataFrame([interpret_response(op, res) for op, res in responses])
print(status_demo[["operation", "status_code", "family", "client_action"]].to_string(index=False))

   operation  status_code family                           client_action
 auth_status          200    2xx                           parse content
bad_endpoint          404    4xx    fix client request or authentication
gateway_down          503    5xx log, pause, and retry only under policy


## 3. One source-style request builder

The source strategy adds `make_request(method, url, headers, payload, qs_params)`. Offline, the same idea is a request record: base URL, endpoint, method, headers, and payload/params are separated before the mock response is processed.

In [3]:
BASE_URL = "https://localhost:5000/v1/api"

def build_request(method, endpoint, headers=None, payload=None, params=None):
    return {
        "base_url": BASE_URL,
        "endpoint": endpoint,
        "url": BASE_URL + "/" + endpoint.lstrip("/"),
        "method": method,
        "headers": headers or {},
        "payload": payload or {},
        "params": params or {},
    }

order_request = build_request(
    "POST",
    "iserver/account/DUXXXX/orders",
    headers={"Content-Type": "application/json"},
    payload={"orders": [{"conid": 56987333, "orderType": "MKT", "side": "BUY", "quantity": 1, "price": 0, "tif": "DAY"}]},
)
print(json.dumps(order_request, indent=2))

{
  "base_url": "https://localhost:5000/v1/api",
  "endpoint": "iserver/account/DUXXXX/orders",
  "url": "https://localhost:5000/v1/api/iserver/account/DUXXXX/orders",
  "method": "POST",
  "headers": {
    "Content-Type": "application/json"
  },
  "payload": {
    "orders": [
      {
        "conid": 56987333,
        "orderType": "MKT",
        "side": "BUY",
        "quantity": 1,
        "price": 0,
        "tif": "DAY"
      }
    ]
  },
  "params": {}
}


## 4. WebSocket message parsing without a socket

REST calls return one response. WebSockets push repeated messages after a subscription such as `smd+contract+fields`. These samples mirror the IB field codes and Alpaca quote shape from the source files.

In [4]:
messages = pd.read_csv(code_dir / "tbp02_mock_websocket_messages.csv")

def parse_ib_market_data(row):
    return {
        "source": row["source"],
        "topic": row["raw_topic"],
        "last_price": float(row["31"]) if pd.notna(row.get("31")) else np.nan,
        "bid": float(row["84"]) if pd.notna(row.get("84")) else np.nan,
        "ask": float(row["86"]) if pd.notna(row.get("86")) else np.nan,
    }

ib_quote = parse_ib_market_data(messages.iloc[0])
alpaca_quote = messages[messages["source"] == "Alpaca"].iloc[0][["symbol", "bid", "ask"]].to_dict()
print("IB parsed quote:", ib_quote)
print("Alpaca parsed quote:", alpaca_quote)

IB parsed quote: {'source': 'IB', 'topic': 'smd+56987333', 'last_price': 18123.5, 'bid': 18122.0, 'ask': 18124.0}
Alpaca parsed quote: {'symbol': 'BTCUSD', 'bid': 63250.1, 'ask': 63252.4}


## 5. Corrected RSI decision table and offline order audit

The source RSI script is preserved as copied source. The addendum corrects the close-position branch by assigning `order_action = 'SELL'` or `order_action = 'BUY'` when exiting existing positions.

In [5]:
decisions = pd.read_csv(code_dir / "tbp02_rsi_strategy_decision_table.csv")
orders = pd.read_csv(code_dir / "tbp02_mock_order_audit.csv")
metrics = pd.read_csv(code_dir / "tbp02_offline_rsi_strategy_metrics.csv")
print(decisions.to_string(index=False))
print("\nMock orders generated:", len(orders))
print(orders[["order_id", "date", "side", "prior_position", "target_position", "reason"]].tail(8).to_string(index=False))
print("\nKey metrics:")
print(metrics[metrics["metric"].isin(["mock_order_count", "offline_rsi_total_return_pct", "buy_hold_total_return_pct", "offline_rsi_max_drawdown_pct"])].to_string(index=False))

current_position         rsi_condition target_position correct_order_action                                                             source_status
               0              RSI < 30               1                  BUY                       source logic already assigns BUY for new long entry
               0              RSI > 70              -1                 SELL                     source logic already assigns SELL for new short entry
               1              RSI > 70               0                 SELL source line uses order_action == SELL; corrected addendum uses assignment
              -1              RSI < 30               0                  BUY  source line uses order_action == BUY; corrected addendum uses assignment
             any threshold not crossed       unchanged                 none                                                     hold current position

Mock orders generated: 13
    order_id       date side  prior_position  target_position            

## 6. Heartbeat schedule

The initial login remains manual. After that, the strategy should keep the gateway alive with periodic `tickle` or reauthentication calls between data and order events.

In [6]:
heartbeat = pd.read_csv(code_dir / "tbp02_heartbeat_schedule.csv")
request_log = pd.read_csv(code_dir / "tbp02_mock_request_log.csv")
print(heartbeat.to_string(index=False))
print("\nStatus families in the mock request log:")
print(request_log.groupby("status_family").size().to_string())

 seconds_from_start                     operation                                                             reason
                  0 manual login already complete keep local gateway session from going idle between strategy events
                 55        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                110        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                165        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                220        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                275        POST /tickle heartbeat keep local gateway session from going idle between strategy events
                330        POST /tickle heartbeat keep local gateway session from going idle between strategy events

Status families in the mock request log:
status_family
2xx    6